# Offensive Language Detection in Tweets

This notebook tackles the three-way classification of tweets: Not Offensive (NOT), Targeted Insult (TIN), and Untargeted profanity (UNT).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 80)

## 1. Load Data

In [ ]:
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')
print(f"Train: {df_train.shape[0]} | Test: {df_test.shape[0]}")
print(df_train['label'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
df_train['label'].value_counts().plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c', '#3498db'])
ax.set_title('Label Distribution')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('nlp_dist_v3.png', dpi=150)
plt.show()

## 2. Preprocessing
We use **Snowball stemmer** (English) for lexical normalisation and **remove stopwords** to focus on content-bearing terms. URLs and @user tokens are stripped.

In [ ]:
stemmer = SnowballStemmer('english')
stop = set(stopwords.words('english'))

def clean(t):
    t = t.lower()
    t = re.sub(r'\burl\b', '', t)
    t = re.sub(r'@user', '', t)
    t = re.sub(r'[^a-zA-Z\s]', '', t)
    tokens = [stemmer.stem(w) for w in t.split() if w not in stop]
    return ' '.join(tokens)

df_train['text_clean'] = df_train['tweet'].apply(clean)
df_test['text_clean'] = df_test['tweet'].apply(clean)
df_train[['tweet', 'text_clean']].head(2)

## 3. Features and Split
**TF-IDF** with `sublinear_tf=True` dampens raw term frequency to reduce bias from very frequent terms. Unigrams only, 8,000 features.

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(
    df_train['text_clean'], df_train['label'],
    test_size=0.3, random_state=100
)

tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 1), sublinear_tf=True, min_df=2)
X_tr_tf = tfidf.fit_transform(X_tr)
X_va_tf = tfidf.transform(X_va)
print(f"Feature matrix: {X_tr_tf.shape}")

## 4. Model Comparison
Three classifiers: Multinomial NB, Gradient Boosting, and K-Nearest Neighbours.

In [ ]:
def run(m, name):
    m.fit(X_tr_tf, y_tr)
    p = m.predict(X_va_tf)
    acc = accuracy_score(y_va, p)
    mf1 = f1_score(y_va, p, average='macro')
    print(f"{name}: Acc={acc:.4f} MacroF1={mf1:.4f}")
    return m, p, acc, mf1

nb, nb_p, nb_a, nb_f = run(MultinomialNB(alpha=0.1), "Multinomial NB")
gb, gb_p, gb_a, gb_f = run(GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=100), "Gradient Boosting")
knn, knn_p, knn_a, knn_f = run(KNeighborsClassifier(n_neighbors=15, weights='distance'), "KNN")

In [ ]:
scores = {'Multinomial NB': (nb_f, nb, nb_p), 'Gradient Boosting': (gb_f, gb, gb_p), 'KNN': (knn_f, knn, knn_p)}
best_name = max(scores, key=lambda k: scores[k][0])
best_f1, best_model, best_preds = scores[best_name]
print(f"\nBest (Macro F1): {best_name}")

## 5. Best Model Analysis

In [ ]:
print(classification_report(y_va, best_preds))
cm = confusion_matrix(y_va, best_preds, labels=best_model.classes_)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=best_model.classes_, yticklabels=best_model.classes_)
plt.title(f'Confusion Matrix - {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('nlp_cm_v3.png', dpi=150)
plt.show()

## 6. Test Predictions

In [ ]:
X_te = tfidf.transform(df_test['text_clean'])
preds = best_model.predict(X_te)
df_test['prediction'] = preds
df_test[['id', 'tweet', 'prediction']].to_csv('test-predictions_v3.csv', index=False)
print(f"Saved {len(df_test)} predictions")
df_test[['id', 'prediction']].head()